# 20 · DPO From First Principles

Companion to **Chapter 24**. You will:
1. Verify the closed-form optimal policy numerically (on a tiny discrete problem
   where you can actually enumerate every completion)
2. Verify that `Z(x)` cancels
3. Implement the loss and check its properties
4. Watch **likelihood displacement** happen

In [ ]:
import os, sys, math
import torch
import torch.nn.functional as F
sys.path.insert(0, os.path.abspath('..'))
from tfs.dpo import dpo_loss

torch.manual_seed(0)

## 1 · The closed-form optimum, verified

Chapter 24.2 claims:

$$\pi^*(y|x) = \frac{1}{Z(x)}\,\pi_{ref}(y|x)\,e^{r(x,y)/\beta}$$

That is a *theorem*, not a heuristic. On a 6-outcome toy problem we can enumerate
everything and check it by brute-force optimisation.

In [ ]:
n = 6
torch.manual_seed(1)
pi_ref = torch.rand(n); pi_ref /= pi_ref.sum()      # reference policy
r = torch.randn(n) * 2                              # some reward
beta = 0.5

# --- the closed form ---
unnorm = pi_ref * torch.exp(r / beta)
Z = unnorm.sum()
pi_star = unnorm / Z

# --- brute force: directly maximise E[r] - beta*KL over the simplex ---
logits = torch.zeros(n, requires_grad=True)
opt = torch.optim.Adam([logits], lr=0.05)
for _ in range(4000):
    pi = logits.softmax(0)
    kl = (pi * (pi / pi_ref).log()).sum()
    objective = (pi * r).sum() - beta * kl
    opt.zero_grad(); (-objective).backward(); opt.step()
pi_opt = logits.softmax(0).detach()

print(f"{'i':>3} {'pi_ref':>9} {'reward':>9} {'closed form':>13} {'brute force':>13}")
for i in range(n):
    print(f"{i:>3} {pi_ref[i]:>9.4f} {r[i]:>9.4f} {pi_star[i]:>13.6f} {pi_opt[i]:>13.6f}")
print(f"\nmax abs diff: {(pi_star - pi_opt).abs().max().item():.2e}")
assert torch.allclose(pi_star, pi_opt, atol=1e-4)
print("The closed form IS the optimum ✓")
print(f"\nZ(x) = {Z.item():.4f}  <- requires summing over ALL completions.")
print("For a real LM that is V^T terms. Intractable. Which is the problem.")

## 2 · Inverting it, and watching Z(x) cancel

$$r(x,y) = \beta\log\frac{\pi^*(y|x)}{\pi_{ref}(y|x)} + \beta\log Z(x)$$

In [ ]:
# Recover r from pi_star -- up to the constant beta*log Z
r_recovered = beta * (pi_star / pi_ref).log() + beta * Z.log()
print("original r :", [f"{v:6.3f}" for v in r.tolist()])
print("recovered  :", [f"{v:6.3f}" for v in r_recovered.tolist()])
assert torch.allclose(r, r_recovered, atol=1e-4)
print("\nThe POLICY encodes the REWARD ✓  ('your LM is secretly a reward model')")

# Now the key step: Bradley-Terry uses only DIFFERENCES
w, l = 2, 5                                   # a preference pair
with_Z    = ((beta*(pi_star[w]/pi_ref[w]).log() + beta*Z.log())
             - (beta*(pi_star[l]/pi_ref[l]).log() + beta*Z.log()))
without_Z = (beta*(pi_star[w]/pi_ref[w]).log()
             - beta*(pi_star[l]/pi_ref[l]).log())
print(f"\nr(y_w) - r(y_l)  WITH the Z terms:    {with_Z.item():.8f}")
print(f"r(y_w) - r(y_l)  WITHOUT the Z terms: {without_Z.item():.8f}")
assert torch.allclose(with_Z, without_Z, atol=1e-6)
print("\nIDENTICAL. Both responses share the same prompt, so both carry the same")
print("beta*log Z(x) and it subtracts away EXACTLY. That is the entire paper.")

## 3 · The loss and its properties

In [ ]:
t = torch.tensor
cases = [
    ("policy strongly prefers chosen",  t([0.0]),  t([-20.0]), t([0.0]), t([0.0])),
    ("policy == reference",             t([-3.0]), t([-5.0]),  t([-3.0]), t([-5.0])),
    ("policy strongly prefers REJECTED",t([-20.0]),t([0.0]),   t([0.0]), t([0.0])),
]
print(f"{'case':>36} {'loss':>9} {'margin':>9} {'acc':>5}")
for name, pc, pr, rc, rr in cases:
    loss, cw, rw, acc = dpo_loss(pc, pr, rc, rr, beta=1.0)
    print(f"{name:>36} {loss.item():>9.5f} {(cw-rw).item():>9.3f} {acc.item():>5.0f}")

print(f"\n'policy == reference' gives loss = -log(0.5) = ln 2 = {math.log(2):.5f}")
print("That is the loss of an untrained-for-preference model -- your step-0 value.")

# Z(x) cancellation, empirically: shift BOTH by any constant
pc, pr, rc, rr = t([-3.0]), t([-7.0]), t([-4.0]), t([-6.0])
base, *_ = dpo_loss(pc, pr, rc, rr, beta=0.1)
print(f"\nbase loss: {base.item():.8f}")
for c in [1.0, -5.0, 100.0]:
    shifted, *_ = dpo_loss(pc+c, pr+c, rc+c, rr+c, beta=0.1)
    print(f"  shift both by {c:>7.1f}: {shifted.item():.8f}")
    assert abs(base.item()-shifted.item()) < 1e-6
print("Invariant to any prompt-level constant ✓  (that IS the Z cancellation)")

## 4 · The gradient does three things

Push chosen up, push rejected down, **weighted by how wrong the model currently is**.

In [ ]:
print(f"{'margin (r_w - r_l)':>20} {'loss':>10} {'|d loss/d margin|':>20}")
for m in [-4.0, -2.0, 0.0, 2.0, 4.0, 8.0]:
    pc = torch.tensor([m], requires_grad=True)
    loss, *_ = dpo_loss(pc, torch.tensor([0.0]),
                        torch.tensor([0.0]), torch.tensor([0.0]), beta=1.0)
    loss.backward()
    print(f"{m:>20.1f} {loss.item():>10.5f} {pc.grad.abs().item():>20.6f}")

print("\nThe gradient weight is sigma(r_l - r_w): near 1 when the model has the pair")
print("BACKWARDS, near 0 when it already ranks them correctly.")
print("Automatic hard-example mining, for free, with no curriculum.")

## 5 · Likelihood displacement — the failure mode to watch for

The loss constrains only the **margin**. It is perfectly happy to push *both*
log-probs down, as long as the rejected one falls faster. Chapter 24 Checkpoint Q3.

In [ ]:
# A toy policy over 5 outcomes; outcome 0 is 'chosen', outcome 1 is 'rejected'.
torch.manual_seed(0)
logits = torch.randn(5, requires_grad=True)
ref_logp = logits.detach().log_softmax(0).clone()
opt = torch.optim.SGD([logits], lr=0.5)

print(f"{'step':>5} {'logp(chosen)':>14} {'logp(rejected)':>16} {'margin':>9} {'loss':>9}")
for step in range(0, 151):
    lp = logits.log_softmax(0)
    loss, cw, rw, _ = dpo_loss(lp[0:1], lp[1:2], ref_logp[0:1], ref_logp[1:2], beta=0.5)
    if step % 30 == 0:
        print(f"{step:>5} {lp[0].item():>14.4f} {lp[1].item():>16.4f} "
              f"{(cw-rw).item():>9.4f} {loss.item():>9.5f}")
    opt.zero_grad(); loss.backward(); opt.step()

final = logits.log_softmax(0)
print(f"\nchosen  log-prob: {ref_logp[0].item():.4f} -> {final[0].item():.4f}")
print(f"rejected log-prob: {ref_logp[1].item():.4f} -> {final[1].item():.4f}")
print(f"probability mass that leaked to the OTHER 3 outcomes: "
      f"{(final[2:].exp().sum() - ref_logp[2:].exp().sum()).item():+.4f}")
print("\nIf the chosen log-prob also fell, you have seen likelihood displacement.")
print("Mitigations: larger beta, fewer epochs, or mix in an SFT loss term.")
print("MONITOR ABSOLUTE LOG-PROBS, not just the margin.")

## 6 · Beta controls the KL leash

In [ ]:
print(f"{'beta':>7} {'final KL(pi||pi_ref)':>22} {'final margin':>14}")
for beta_ in [0.01, 0.1, 0.5, 1.0]:
    torch.manual_seed(0)
    lg = torch.randn(5, requires_grad=True)
    ref = lg.detach().log_softmax(0).clone()
    o = torch.optim.SGD([lg], lr=0.5)
    for _ in range(300):
        lp = lg.log_softmax(0)
        loss, cw, rw, _ = dpo_loss(lp[0:1], lp[1:2], ref[0:1], ref[1:2], beta=beta_)
        o.zero_grad(); loss.backward(); o.step()
    lp = lg.log_softmax(0).detach()
    kl = (lp.exp() * (lp - ref)).sum().item()
    print(f"{beta_:>7} {kl:>22.4f} {(cw-rw).item():>14.4f}")

print("\nSmaller beta -> weaker leash -> larger KL -> more drift from the SFT model.")
print("beta=0.01 is where real models start producing degenerate, terse output.")
print("beta=0.1 is the standard starting point. This is the SAME KL coefficient")
print("as in the PPO objective -- DPO inherited it from the derivation.")

---
## Self-check

1. Why does `Z(x)` cancel? What property of the pair makes it work?
2. What does `beta` control?
3. Why sum log-probs rather than average them?
4. Both chosen and rejected log-probs are falling. Bug or normal?
5. What is DPO's main structural limitation vs PPO?

<details><summary>Answers</summary>

1. `Z(x)` depends only on the **prompt**. Both responses in a preference pair share
   that prompt, so the identical `β log Z(x)` term appears in both rewards and
   subtracts away exactly in the Bradley-Terry difference.
2. The KL constraint strength — how far the policy may drift from `π_ref`. Same
   coefficient as PPO's KL penalty. Not a learning rate.
3. The derivation is about `log π(y|x) = Σ_t log π(y_t|·)`, a sum. Averaging is a
   different (deliberate) objective — that's SimPO, which uses it to remove length bias.
4. **Normal**, and widely reported. The loss only constrains the margin. But watch it:
   pushed too far it becomes likelihood displacement and quality degrades.
5. DPO is **off-policy** — it learns from a fixed dataset generated by some other
   model. As the policy drifts, that data describes behaviour it no longer has.
   Iterative/online DPO recovers much of this.

</details>

**Next:** the capstone — `course/ch26-capstone.html`